In [1]:
import pandas as pd

In [ ]:
#url = "https://raw.githubusercontent.com/vijayaideveloper/CoffeeShopRevenuePrediction/refs/heads/main/dataset/coffee_shop_dataset.csv"

#df = pd.read_csv(url)
#df.head(10)

In [2]:
# if local machine
data_path = "../dataset/coffee_shop_dataset.csv"
df = pd.read_csv(data_path)
df.head(10)

,Number_of_Customers_Per_Day,Average_Order_Value,Marketing_Spend_Per_Day,Daily_Revenue
0,152,6.74,106.62,1547.81
1,485,4.50,57.83,2084.68
2,398,9.09,91.76,3118.39
3,320,8.48,462.63,2912.20
4,156,7.44,412.52,1663.42
5,121,8.88,183.49,1155.18
6,238,9.00,331.35,2179.13
7,70,7.81,273.27,890.17
8,152,8.78,341.79,1704.94
9,171,7.73,344.51,2025.55


In [3]:
import torch
from torch import nn
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [5]:
X = df.drop('Daily_Revenue', axis=1)
y = df['Daily_Revenue']

X.head()

,Number_of_Customers_Per_Day,Average_Order_Value,Marketing_Spend_Per_Day
0,152,6.74,106.62
1,485,4.50,57.83
2,398,9.09,91.76
3,320,8.48,462.63
4,156,7.44,412.52


In [6]:
y[:10]

0    1547.81
1    2084.68
2    3118.39
3    2912.20
4    1663.42
5    1155.18
6    2179.13
7     890.17
8    1704.94
9    2025.55
Name: Daily_Revenue, dtype: float64

In [7]:
X.shape, y.shape

((2000, 3), (2000,))

In [8]:
X.ndim, y.ndim

(2, 1)

In [9]:
X_tensor = torch.tensor(X.values, dtype=torch.float32).to(device)
y_tensor = torch.tensor(y.values, dtype=torch.float32).reshape(-1, 1).to(device)

X_tensor[:5], y_tensor[:5]

(tensor([[152.0000,   6.7400, 106.6200],
         [485.0000,   4.5000,  57.8300],
         [398.0000,   9.0900,  91.7600],
         [320.0000,   8.4800, 462.6300],
         [156.0000,   7.4400, 412.5200]]),
 tensor([[1547.8101],
         [2084.6799],
         [3118.3899],
         [2912.2000],
         [1663.4200]]))

In [10]:
X_tensor.ndim, y_tensor.ndim

(2, 2)

In [11]:
X_tensor.shape, y_tensor.shape

(torch.Size([2000, 3]), torch.Size([2000, 1]))

In [12]:
X_scaler = StandardScaler()
y_scaler = StandardScaler()

X_tensor = torch.tensor(
    X_scaler.fit_transform(X_tensor.cpu().numpy()),
    dtype=torch.float32
).to(device)

y_tensor = torch.tensor(
    y_scaler.fit_transform(y_tensor.cpu().numpy()),
    dtype=torch.float32
).to(device)

In [13]:
X_tensor[0], y_tensor[0]

(tensor([-0.9450,  0.2201, -1.0347]), tensor([-0.3786]))

In [14]:
split_pct = 0.8 

split_size = int(len(X_tensor) * split_pct)

X_train = X_tensor[:split_size]
y_train = y_tensor[:split_size]

X_test = X_tensor[split_size:]
y_test = y_tensor[split_size:]

X_train.shape, X_test.shape, y_train.shape, y_test.shape

(torch.Size([1600, 3]),
 torch.Size([400, 3]),
 torch.Size([1600, 1]),
 torch.Size([400, 1]))

In [15]:
X_train = X_train.to(device)
y_train = y_train.to(device)

X_test = X_test.to(device)
y_test = y_test.to(device)

In [16]:
torch.manual_seed(42)

class CoffeeShopRevenueV0(nn.Module):

    def __init__(self):

        super().__init__()

        self.linear_layer = nn.Sequential(
            nn.Linear(3, 1)
        )

    def forward(self, x):
        return self.linear_layer(x)


model0 = CoffeeShopRevenueV0().to(device)
print(model0)

CoffeeShopRevenueV0(
  (linear_layer): Sequential(
    (0): Linear(in_features=3, out_features=1, bias=True)
  )
)


In [67]:
learning_rate = 0.001

loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(params=model0.parameters(),
                            lr=learning_rate)

In [71]:
torch.manual_seed(42)

epochs = 100

for epoch in range(epochs):
    model0.train()

    y_predicted = model0(X_train).to(device)

    loss = loss_fn(y_train, y_predicted)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    with torch.inference_mode():
        y_test_pred = model0(X_test).to(device)

        test_loss = loss_fn(y_test, y_test_pred)

        if epoch % 10 == 0:
            print(f"epoch {epoch} | Train loss : {loss.item():.4f} | Test Loss {test_loss.item():.4f}")

epoch 0 | Train loss : 0.1082 | Test Loss 0.1102
epoch 10 | Train loss : 0.1082 | Test Loss 0.1102
epoch 20 | Train loss : 0.1082 | Test Loss 0.1102
epoch 30 | Train loss : 0.1082 | Test Loss 0.1102
epoch 40 | Train loss : 0.1082 | Test Loss 0.1102
epoch 50 | Train loss : 0.1082 | Test Loss 0.1102
epoch 60 | Train loss : 0.1082 | Test Loss 0.1102
epoch 70 | Train loss : 0.1082 | Test Loss 0.1102
epoch 80 | Train loss : 0.1082 | Test Loss 0.1102
epoch 90 | Train loss : 0.1082 | Test Loss 0.1102


In [72]:
X_test[2], y_test[2]

(tensor([ 0.5232,  0.9648, -1.0626]), tensor([0.5634]))

In [88]:
model0.eval()
with torch.inference_mode():
    prediction = model0(X_test[2]).to(device)
    print("scaler value : ", prediction.item())
    prediction = y_scaler.inverse_transform(
        prediction.cpu().numpy().reshape(1, -1)).item()

print('Prediction : ', prediction)



scaler value :  0.677198052406311
Prediction :  2578.2431640625
